# 7조 작업 노트북 — 듣고, 보고, 비교하기

강의 배포 노트북(`dataset_exploration`, `noise_reduction_baselines`)의 작업 방식을 **저장소 코드 그대로** 쓰도록 옮긴 노트북입니다.
Colab(데이터는 Google Drive)과 로컬 Jupyter(conda `denoise` 환경) 모두에서 위에서 아래로 실행합니다.

- 알고리즘은 `nr/`의 `enhance()`를 호출하므로 **제출 코드(`denoise.py`)와 같은 결과**가 나옵니다. 노트북 안에서 알고리즘을 따로 고치지 마세요 — 고칠 곳은 `nr/`와 `configs/`입니다.
- Clean 음성은 비교·점수 계산에만 씁니다.
- 노트북 출력은 커밋하지 않습니다 (커밋 전 *출력 모두 지우기*).

| 절 | 내용 | 강의 노트북 대응 |
|---|---|---|
| 1 | 환경: 저장소 clone/설치, Drive 연결 | 1·2절 |
| 2 | 데이터 개요 | dataset_exploration 2·3절 |
| 3 | 샘플 하나 골라 추론·스펙트로그램·청취 | 7·10절, dataset_exploration 4·5절 |
| 4 | 빠른 평가(train quick)와 validation 평가 | 8·12절 |

## 1. 실행 환경

Colab에서는 팀 저장소를 `/content/-7-`에 clone하고 `requirements-dev.txt`를 설치합니다(pesq는 Linux에서 바로 설치됩니다).
로컬에서는 이 노트북이 있는 저장소를 그대로 씁니다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/gaga00-gaga00/-7-.git"
BRANCH = "main"  # 특정 제출 회차를 보려면 "submission-1" 같은 태그

if IN_COLAB:
    REPO = Path("/content/-7-")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "--tags"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO / "requirements-dev.txt"), "ipywidgets>=8,<9"], check=True)
    from google.colab import output
    output.enable_custom_widget_manager()
else:
    # local Jupyter: find the repository root above this notebook
    REPO = Path.cwd().resolve()
    while not (REPO / "denoise.py").is_file() and REPO != REPO.parent:
        REPO = REPO.parent
    # conda DLLs (libsndfile) when the kernel starts without `conda activate`
    if sys.platform == "win32" and (Path(sys.prefix) / "conda-meta").is_dir():
        conda_bin = Path(sys.prefix) / "Library" / "bin"
        os.environ["PATH"] = str(conda_bin) + os.pathsep + os.environ.get("PATH", "")
        _dll = os.add_dll_directory(str(conda_bin))

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Python", sys.version.split()[0], "| repo", REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

### 데이터 경로

- **Colab:** 배포 폴더(`train/`, `validation/`, `figures/`)를 Drive에 올리고 `GOOGLE_DRIVE_DATA_PATH`를 각자 위치로 고치세요. 저장소 안에 `train`, `validation` 링크를 만들어 `evaluate.py` 등 모든 도구가 그대로 동작하게 합니다.
- **로컬:** 저장소 루트에 `train/`, `validation/`을 두면 됩니다.

In [ ]:
GOOGLE_DRIVE_DATA_PATH = Path("/content/drive/MyDrive/ColabNotebooks/신시/noise_reduction_challenge")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    for split in ("train", "validation", "figures"):
        link, target = REPO / split, GOOGLE_DRIVE_DATA_PATH / split
        if target.exists() and not link.exists():
            link.symlink_to(target, target_is_directory=True)

from evaluate import load_manifest

DATASETS = {split: load_manifest(REPO, split, None) for split in ("validation", "train")}
print({split: len(df) for split, df in DATASETS.items()})

### 설정

제출 버전 설정(`configs/submission.json`)을 읽고, 비교할 방법을 고릅니다. 다른 설정 파일을 보려면 `CONFIG_FILE`을 바꾸세요.

In [ ]:
import numpy as np
import pandas as pd

from nr import METHODS, enhance, load_config

CONFIG_FILE = REPO / "configs" / "submission.json"
SUBMIT_METHOD, CFG = load_config(CONFIG_FILE)
SHOW_METHODS = ["basic_filter", "spectral_subtraction", "wiener_filter", SUBMIT_METHOD]

print("등록된 방법:", ", ".join(METHODS))
print("설정:", CONFIG_FILE.name, "→", SUBMIT_METHOD)
print("비교할 방법:", SHOW_METHODS)

## 2. 데이터 개요

분할별 화자·길이·잡음 종류, 잡음 종류별 샘플 수, 난이도×SNR 분포를 `tools/explore_dataset.py`로 계산합니다.
Validation 화자는 train에 없습니다(히든 테스트셋도 새로운 화자·잡음).

In [ ]:
from IPython.display import Image, display

subprocess.run([sys.executable, "tools/explore_dataset.py"], check=True)
display(Image(filename=str(REPO / "results" / "figures" / "dataset" / "noise_counts.png"), width=900))

## 3. 샘플 하나: 추론 → 스펙트로그램 → 청취

Split·Noise·Code·SNR·Difficulty·Sample을 고르고 버튼을 누르면 그 파일 하나만 처리합니다.
**파일 경로** 모드에서는 noisy 경로(필수)와 clean 경로(선택)를 직접 입력합니다(상대 경로는 저장소 기준).

- 모든 스펙트로그램은 **noisy STFT 최대값을 0 dB**로 같은 색 범위를 씁니다.
- 재생 버튼은 모두 **같은 gain**이라 방법 간 크기 차이가 그대로 들립니다.
- `added noise = noisy − clean`은 섞인 잡음만 보여 줍니다(clean이 있을 때).
- Enhanced는 추정 결과이지 정답이 아닙니다.

In [ ]:
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, Markdown, display

from evaluate import pesq_wb, snr_db
from tools.spectrogram import stft_mag


def show_example(noisy_path, clean_path=None, methods=None, cfg=None, added_noise=True,
                 title=None):
    """Enhance one file with each method; plot spectrograms and add audio players."""
    methods = methods or SHOW_METHODS
    cfg = cfg or CFG
    noisy, sr = sf.read(str(REPO / noisy_path), dtype="float64")
    if noisy.ndim != 1:
        noisy = noisy.mean(axis=1)
    clean = None
    if clean_path:
        clean, sr_c = sf.read(str(REPO / clean_path), dtype="float64")
        if sr_c != sr or clean.shape != noisy.shape:
            raise ValueError("clean과 noisy의 표본률·길이가 같아야 합니다.")
    panels = ([("clean", clean)] if clean is not None else []) + [("noisy", noisy)]
    if added_noise and clean is not None:
        panels.append(("added noise", noisy - clean))
    panels += [(m, enhance(noisy, sr, m, cfg)) for m in methods]

    ref = max(float(stft_mag(noisy, sr)[2].max()), 1e-12)
    fig, axes = plt.subplots(len(panels), 1, figsize=(12, 2.1 * len(panels)), sharex=True,
                             sharey=True, constrained_layout=True)
    rows = []
    for ax, (name, x) in zip(axes, panels):
        f, t, mag = stft_mag(x, sr)
        mesh = ax.pcolormesh(t, f / 1000, 20 * np.log10(np.maximum(mag, 1e-12) / ref),
                             shading="auto", cmap="magma", vmin=-80, vmax=0)
        label = name
        if clean is not None and name not in ("clean", "added noise"):
            p, s = pesq_wb(clean, x, sr), snr_db(clean, x)
            label += f"   PESQ {p:.2f} · SNR {s:.1f} dB"
            rows.append({"method": name, "pesq": p, "snr_db": s})
        ax.set_title(label, loc="left", fontsize=10)
        ax.set_ylabel("kHz")
    axes[-1].set_xlabel("Time (s)")
    fig.colorbar(mesh, ax=list(axes), label="dB re noisy STFT peak", shrink=0.8)
    fig.suptitle(title or Path(noisy_path).name)
    display(fig)
    plt.close(fig)
    if rows:
        display(pd.DataFrame(rows).set_index("method").round(3))

    gain = 0.99 / max(1.0, max(float(np.abs(x).max()) for _, x in panels))
    display(Markdown(f"공통 재생 gain: {gain:.4f}"))
    for name, x in panels:
        display(Markdown(f"**{name}**"))
        display(Audio(x * gain, rate=sr, normalize=False))
    return dict(panels)

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output


def select_examples(manifest, noise_category=None, noise_code=None, target_snr_db=None,
                    noise_difficulty=None):
    """Filter metadata rows only (no audio is read)."""
    selected = manifest
    for column, value in (("noise_category", noise_category), ("noise_code", noise_code),
                          ("noise_difficulty", noise_difficulty)):
        if value is not None:
            selected = selected[selected[column].eq(value)]
    if target_snr_db is not None:
        selected = selected[np.isclose(selected.target_snr_db, float(target_snr_db))]
    return selected.sort_values("segment_id").reset_index(drop=True)


def create_sample_browser(datasets, button_label="추론 / 시각화 / 청취"):
    all_rows = pd.concat(datasets.values(), ignore_index=True)
    opt = lambda values, fmt=str: [("전체", None)] + [(fmt(v), v) for v in sorted(values)]
    split = widgets.Dropdown(options=list(datasets), value="validation", description="Split:")
    noise = widgets.Dropdown(options=opt(all_rows.noise_category.unique()), description="Noise:")
    code = widgets.Dropdown(options=opt(all_rows.noise_code.unique()), description="Code:")
    snr = widgets.Dropdown(options=opt(all_rows.target_snr_db.astype(float).unique(),
                                       lambda v: f"{v:g} dB"), description="SNR:")
    difficulty = widgets.Dropdown(options=opt(all_rows.noise_difficulty.unique()),
                                  description="Difficulty:")
    sample = widgets.Dropdown(description="Sample:", layout=widgets.Layout(width="95%"))
    methods = widgets.SelectMultiple(options=list(METHODS), value=tuple(SHOW_METHODS),
                                     description="Methods:", rows=6)
    mode = widgets.ToggleButtons(options=["데이터셋", "파일 경로"], description="선택 방식:")
    noisy_path = widgets.Text(description="Noisy:", placeholder="noisy WAV 경로 (저장소 기준)",
                              layout=widgets.Layout(width="95%"))
    clean_path = widgets.Text(description="Clean:", placeholder="선택: 대응하는 clean WAV",
                              layout=widgets.Layout(width="95%"))
    filters = widgets.VBox([widgets.HBox([split, noise]), widgets.HBox([code, snr, difficulty]),
                            sample])
    paths = widgets.VBox([noisy_path, clean_path])
    status = widgets.Label()
    run = widgets.Button(description=button_label, button_style="primary",
                         layout=widgets.Layout(width="220px"))
    output = widgets.Output()
    state = {"candidates": pd.DataFrame()}

    def sync(change=None):
        path_mode = mode.value == "파일 경로"
        filters.layout.display = "none" if path_mode else ""
        paths.layout.display = "" if path_mode else "none"
        ready = bool(noisy_path.value.strip()) if path_mode else not state["candidates"].empty
        run.disabled = not ready
        status.value = ("Noisy 경로를 입력하세요." if path_mode else
                        f"조건에 맞는 샘플 {len(state['candidates'])}개")

    def refresh(change=None):
        state["candidates"] = select_examples(
            datasets[split.value], noise_category=noise.value, noise_code=code.value,
            target_snr_db=snr.value, noise_difficulty=difficulty.value)
        sample.options = [(f"{i}: {r.segment_id} | {r.noise_category} {r.noise_code} | "
                           f"{r.target_snr_db:g} dB | {r.noise_difficulty}", i)
                          for i, r in state["candidates"].iterrows()]
        sample.value = 0 if len(state["candidates"]) else None
        sync()

    def go(button=None):
        run.disabled = True
        try:
            with output:
                clear_output(wait=True)
                try:
                    if mode.value == "파일 경로":
                        show_example(noisy_path.value.strip(), clean_path.value.strip() or None,
                                     list(methods.value))
                    elif sample.value is not None:
                        r = state["candidates"].iloc[sample.value]
                        show_example(r.noisy_path, r.clean_path, list(methods.value),
                                     title=f"{r.segment_id} · {r.noise_category} "
                                           f"({r.noise_code}, {r.noise_difficulty}) · "
                                           f"{r.target_snr_db:g} dB")
                except (ValueError, FileNotFoundError, RuntimeError) as error:
                    print(f"입력을 확인하세요: {error}")
        finally:
            sync()

    for control in (split, noise, code, snr, difficulty):
        control.observe(refresh, names="value")
    for control in (sample, mode, noisy_path, clean_path):
        control.observe(sync, names="value")
    run.on_click(go)
    refresh()
    return widgets.VBox([mode, filters, paths, methods, status, run, output])


display(create_sample_browser(DATASETS))

위젯이 보이지 않으면 직접 호출하세요.

```python
row = select_examples(DATASETS["validation"], noise_category="wind").iloc[0]
_ = show_example(row.noisy_path, row.clean_path)
```

## 4. 평가

노트북에서 따로 평가 루프를 만들지 않고 `evaluate.py`를 실행합니다(강의 노트북과 같은 정의: PESQ-WB, 정규화·정렬 없음, 파일별 평균).
결과는 `results/runs/<실행 id>/`와 `results/experiments.csv`에 쌓입니다.

- 데이터는 전부 씁니다: **train quick(200개)으로 빠르게 튜닝 → train 전체(1,296개)로 후보 확인 → validation 전체(164개)로 최종 확인.**
- `SPLIT_ARGS`: quick `["--split", "train", "--subset", "quick"]` · train 전체 `["--split", "train"]` · validation 전체 `[]` · 둘 다 `["--split", "all"]`
- `SET`에 `"lsa_gain_floor=0.1"`처럼 파라미터를 바꿔 볼 수 있습니다.
- Colab에서 만든 결과는 Colab 안의 clone에만 남습니다. 남기려면 `experiments.csv`를 커밋·push하거나 결과 표를 팀에 공유하세요.

In [ ]:
SPLIT_ARGS = ["--split", "train", "--subset", "quick"]   # train 전체: ["--split", "train"], validation: [], 둘 다: ["--split", "all"]
METHODS_TO_SCORE = ["wiener_filter", SUBMIT_METHOD]
SET = []                                                   # 예: ["lsa_gain_floor=0.1"]
TAG = "colab" if IN_COLAB else "notebook"
NOTE = ""                                                  # 무엇을 바꿨는지

cmd = [sys.executable, "evaluate.py", *SPLIT_ARGS, "--config", str(CONFIG_FILE),
       "--methods", *METHODS_TO_SCORE, "--tag", TAG, "--note", NOTE]
if SET:
    cmd += ["--set", *SET]
print("$", " ".join(cmd[1:]))
subprocess.run(cmd, check=True)

마지막 두 실행을 파일별로 비교합니다(평균만 보지 말고 잡음 종류·SNR별 손해 그룹을 확인).

In [ ]:
runs = sorted((REPO / "results" / "runs").iterdir())
print("최근 실행:", [r.name for r in runs[-5:]])
if runs:
    last = runs[-1]
    if not (last / "per_file.csv").exists():  # --split all: one folder per split
        last = last / "train"
    methods = pd.read_csv(last / "per_file.csv").method.unique()
    others = [m for m in methods if m != "noisy_input"]
    if len(others) >= 2:
        subprocess.run([sys.executable, "compare.py", str(last), str(last),
                        "--method-a", others[0], "--method-b", others[-1]], check=True)